### <font style="color:blue">Project 2: Kaggle Competition - Classification</font>

#### Maximum Points: 100

<div>
    <table>
        <tr><td><h3>Sr. no.</h3></td> <td><h3>Section</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1</h3></td> <td><h3>Data Loader</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>2</h3></td> <td><h3>Configuration</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>3</h3></td> <td><h3>Evaluation Metric</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>4</h3></td> <td><h3>Train and Validation</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>5</h3></td> <td><h3>Model</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>6</h3></td> <td><h3>Utils</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>7</h3></td> <td><h3>Experiment</h3></td><td><h3>5</h3></td> </tr>
        <tr><td><h3>8</h3></td> <td><h3>TensorBoard Dev Scalars Log Link</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>9</h3></td> <td><h3>Kaggle Profile Link</h3></td> <td><h3>50</h3></td> </tr>
    </table>
</div>


## <font style="color:green">1. Data Loader [10 Points]</font>

In this section, you have to write a class or methods, which will be used to get training and validation data loader.

You need to write a custom dataset class to load data.

**Note; There is   no separate validation data. , You will thus have to create your own validation set, by dividing the train data into train and validation data. Usually, we do 80:20 ratio for train and validation, respectively.**


For example:

```python
class KenyanFood13Dataset(Dataset):
    """
    
    """
    
    def __init__(self, *args):
    ....
    ...
    
    def __getitem__(self, idx):
    ...
    ...
    

```


```python
def get_data(args1, *args):
    ....
    ....
    return train_loader, test_loader
```

In [ ]:
import copy
from dataclasses import dataclass
import math
import numpy as np
import pandas as pd
import os
import random
import requests
import time
from typing import List, Union, Tuple
import warnings
from zipfile import ZipFile, BadZipFile
import zipfile

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset
from torch.optim import lr_scheduler
from torch.utils.data import random_split
from tqdm import tqdm
from torchinfo import summary
from torchmetrics import MeanMetric
from torchmetrics.classification import MulticlassAccuracy

from torch.optim.lr_scheduler import OneCycleLR

from torchvision.models import resnet50
from torchvision import datasets, transforms as T
from torchvision.transforms import functional as FU
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Subset

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter)
from PIL import Image
import sklearn

# To filter UserWarning.
warnings.filterwarnings("ignore", category=UserWarning)

bold = f"\033[1m"
reset = f"\033[0m"

In [ ]:
def system_config(SEED_VALUE=52, package_list=None):
    """
    Configures the system environment for PyTorch-based operations.

    Args:
        SEED_VALUE (int): Seed value for random number generation. Default is 42.
        package_list (str): String containing a list of additional packages to install
        for Google Colab or Kaggle. Default is None.

    Returns:
        tuple: A tuple containing the device name as a string and a boolean indicating GPU availability.
    """

    random.seed(SEED_VALUE)
    np.random.seed(SEED_VALUE)
    torch.manual_seed(SEED_VALUE)

    def is_running_in_colab():
        return 'COLAB_GPU' in os.environ

    def is_running_in_kaggle():
        return 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

    #--------------------------------
    # Check for availability of GPUs.
    #--------------------------------
    if torch.cuda.is_available():
        print('Using CUDA GPU')

        # This section for installing packages required by Colab.
        if is_running_in_colab() or is_running_in_kaggle():
            print('Installing required packages...')
            !pip install {package_list}

        # Set the device to the first CUDA device.
        DEVICE = torch.device('cuda')
        print("Device: ", DEVICE)
        GPU_AVAILABLE = True
        
        torch.cuda.manual_seed(SEED_VALUE)
        torch.cuda.manual_seed_all(SEED_VALUE)

        # Performance and deterministic behavior.
        torch.backends.cudnn.enabled = True       # Provides highly optimized primitives for DL operations.
        torch.backends.cudnn.deterministic = True # Insures deterministic even when above cudnn is enabled.
        torch.backends.cudnn.benchmark = False    # Setting to True can cause non-deterministic behavior.

    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        print('Using Apple Silicon GPU')

        # Set the device to the Apple Silicon GPU Metal Performance Shader (MPS).
        DEVICE = torch.device("mps")
        print("Device: ", DEVICE)
        # Environment variable that allows PyTorch to fall back to CPU execution
        # when encountering operations that are not currently supported by MPS.
        os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
        GPU_AVAILABLE = True

        torch.mps.manual_seed(SEED_VALUE)
        torch.use_deterministic_algorithms(True)

    else:
        print('Using CPU')
        DEVICE = torch.device('cpu')
        print("Device: ", DEVICE)
        GPU_AVAILABLE = False

        if is_running_in_colab() or is_running_in_kaggle():
            print('Installing required packages...')
            !pip install {package_list}
            print('Note: Change runtime type to GPU for better performance.')

        torch.use_deterministic_algorithms(True)

    return str(DEVICE), GPU_AVAILABLE

package_list = "torchmetrics torchinfo tqdm"
DEVICE, GPU_AVAILABLE = system_config(package_list=package_list)

In [ ]:
archive_name = "open-cv-py-torch-project-2-classification-round-4"
dataroot = "/kaggle/input"
class_to_idx = {'bhaji': 0, 'chapati': 1, 'githeri': 2, 'kachumbari': 3, 'kukuchoma': 4, 'mandazi': 5, 'masalachips': 6, 'matoke': 7, 'mukimo': 8, 'nyamachoma': 9, 'pilau': 10, 'sukumawiki': 11, 'ugali': 12}
idx_to_class = {v:k for k,v in class_to_idx.items()}

In [ ]:
class FoodDataset(Dataset):

    def __init__(self,data_root,class_dict,train=True, image_shape = None, transform = None):
        self.data_root = data_root
        self.img_dir = os.path.join(data_root,'images/images')
        self.transform = transform
        self.data_dict = None
        self.label_df = None
        self.train = train
        num_classes = 13
        self.class_to_idx = class_dict
        if self.train:
            data_df = 'train.csv'
            self.label_df = pd.read_csv(os.path.join(data_root,data_df))
            self.data_dict = {
                "image_path": list(self.label_df["id"].values),
                "label": list(self.label_df["class"].values)
            }
        else:
            data_df = 'test.csv'
            self.label_df = pd.read_csv(os.path.join(data_root,data_df))
            self.data_dict = {
                "image_path": list(self.label_df["id"].values)
            }
        if image_shape is not None:
            if isinstance(image_shape,int):
                self.image_shape = (image_shape,image_shape)
            elif isinstance(image_shape, tuple) or isinstance(image_shape, list):
                assert len(image_shape) == 1 or len(image_shape) == 2, 'Invalid image_shape tuple size'
                if len(image_shape) == 1:
                    self.image_shape = (image_shape[0], image_shape[0])
                else:
                    self.image_shape = image_shape
            else:
                raise NotImplementedError
        else:
            self.image_shape = image_shape



    def __len__(self):
        return len(self.data_dict['image_path'])

    
    def __getitem__(self, idx):
        image_id = self.data_dict['image_path'][idx]          # get the filename / image ID
        image = Image.open(os.path.join(self.img_dir, f"{image_id}.jpg")).convert("RGB")
    
        if self.image_shape is not None:
            image = FU.resize(image, self.image_shape)
    
        if self.transform is not None:
            image = self.transform(image)
        if self.train:
            class_name = self.data_dict['label'][idx]
            target = torch.tensor(self.class_to_idx[class_name], dtype=torch.long)
            return image, target
        else:
            return image, str(image_id)


In [ ]:
def get_data(img_size,dataset, batch_size, train_val_split=.8, shuffle_val_data=False, num_workers=2,test=False):

    mean=[0.485, 0.456, 0.406]
    std= [0.229, 0.224, 0.225]

    common_transforms = T.Compose([
                            T.Resize(img_size),
                            T.ToTensor(),
                            T.Normalize(mean=mean, std=std)
                        ])

    if test:
        dataset = FoodDataset(data_root = dataset.data_root,train=False,transform = common_transforms, class_dict = class_to_idx)
        test_loader = DataLoader(dataset,batch_size,num_workers=num_workers)
        print("testing")
        return test_loader
    else:
        train_transforms = T.Compose([
                                T.RandomResizedCrop(img_size,scale=(0.8,1.0)),
                                T.RandomHorizontalFlip(p=0.5),
                                T.RandomRotation(degrees=20),
                                T.ColorJitter(.2,.2,.2,.05), 
                                T.ToTensor(),
                                T.Normalize(mean=mean, std=std)
                           ])
        indices = torch.randperm(len(dataset))
        
        train_size = int(train_val_split*len(dataset))
        
        train_indices = indices[:train_size]
        val_indices = indices[train_size:]

        
        
        train_dataset = FoodDataset(data_root=dataset.data_root, transform=train_transforms,class_dict = class_to_idx        )
    
        val_dataset = FoodDataset(data_root=dataset.data_root,transform=common_transforms,class_dict = class_to_idx)
    
        train_set = Subset(train_dataset, train_indices)
        val_set   = Subset(val_dataset, val_indices)
        
        # Train dataloader.
        train_loader = DataLoader(train_set, 
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=num_workers)
    
        # Test dataloader.
        valid_loader = DataLoader(val_set, 
                                 batch_size=batch_size,
                                 shuffle=shuffle_val_data,
                                 num_workers=num_workers)
        return train_loader, valid_loader

In [ ]:
def collate_fn(batch):
    images, ids = zip(*batch)  # separate images and IDs
    images = torch.stack(images, 0)  # stack the image tensors
    return images, ids  # keep IDs as a tuple (don’t try to convert to tensor)

preprocess = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor()
    ])

In [ ]:
data_root = os.path.join(dataroot, "open-cv-py-torch-project-2-classification-round-4")

train_dataset = FoodDataset(data_root, class_to_idx,train = True, image_shape = 256,transform=preprocess)

print("Length of Dataset: {}".format(len(train_dataset)))

img,trgt = train_dataset[300]

print("label: {}".format(trgt))
img = img.permute(1,2,0)
plt.imshow(img)
plt.show()



In [ ]:
test_dataset = FoodDataset(data_root, class_to_idx,train=False, image_shape=None,transform=preprocess)
dataset_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=15,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)


plt.rcParams["figure.figsize"]=(15,9)
plt.figure
for images, labels in dataset_loader:
    for i in range(len(labels)):
        plt.subplot(3, 5, i + 1)
        img = FU.to_pil_image(images[i])
        plt.imshow(img)
        plt.gca().set_title(f'ID: {labels[i]}')  # labels[i] is the image_id
    plt.show()
    break

## <font style="color:green">2. Configuration [5 Points]</font>

**Define your configuration here.**

For example:


```python
@dataclass
class TrainingConfiguration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 10 
    epochs_count: int = 50  
    init_learning_rate: float = 0.1  # initial learning rate for lr scheduler
    log_interval: int = 5  
    test_interval: int = 1  
    data_root: str = "/kaggle/input/opencv-pytorch-project-2-classification-round-3" 
    num_workers: int = 2  
    device: str = 'cuda'  
    
```

In [ ]:
@dataclass(frozen=True)
class TrainingConfiguration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 32 
    EPOCHS: int = 15
    LEARNING_RATE: float = 3e-4  # initial learning rate for lr scheduler
    MOMENTUM:       float = 0.9
    log_interval: int = 5  
    test_interval: int = 1  
    data_root: str = "/kaggle/input/opencv-pytorch-project-2-classification-round-4" 
    num_workers: int = 2  
    device: str = 'cuda'  
    LOG_DIR:        str = "/kaggle/working/FOOD_DIR"
    CHECKPOINT_DIR: str = "/kaggle/working/FOOD_CHECKPOINTS"



In [ ]:
@dataclass(frozen=True)
class DatasetConfiguration:
    NUM_CLASSES: int = 13
    IMG_HEIGHT:  int = 224
    IMG_WIDTH:   int = 224
    NUM_WORKERS: int = 0
    DATA_ROOT:   str = archive_name



## <font style="color:green">3. Evaluation Metric [10 Points]</font>

**Define methods or classes that will be used in model evaluation. For example, accuracy, f1-score etc.**

In [ ]:
class ConfusionMatrix:
    def __init__(self):
        self.conf = np.ndarray((2,2),np.int32)

    def reset(self):
        self.conf.fill(0)

    def add(self,pred,target):
        replace_indices = np.vstack((target.flatten(),pred.flatten())).T
        conf, _ = np.histogramdd(replace_indices, bins=(2,2), range=[(0,2),(0,2)])
        self.conf += conf.astype(np.int32)

    def TP(self):
        return self.conf[1,1]
    def FP(self):
        return self.conf[0,1]
    def TN(self):
        return self.conf[0,0]
    def TP(self):
        return self.conf[1,0]

    def confusion_matrix(self):
        cm = np.array([[self.TP(), self.FP()],
                      [self.FN(), self.TN()]])
        return cm

In [ ]:
def precision(thresh_prob, y_predicted,y_true):
    predictions = y_predicted>thresh_prob
    cm.reset()
    cm.add(predictions,y_true)

def recall(thresh_prob,y_predicted,y_true):
    predictions = y_predicted, y_true
    cm.reset()
    cm.add(predictions,y_true)
    rec = cm.TP()/(cm.TP()+cm.FN())

    return rec

def accuracy(thresh_prob, y_predicted, y_true):
    predictions = y_predicted>thresh_prob
    cm.reset()
    cm.add(predictions,y_true)
    acc = (cm.TP()+cm.TN())/(cm.TP()+cm.FP()+cm.FN+cm.TN)

def f1_score(thresh_prob,y_predicted, y_true):
    predictions = y_predicted>thresh_prob

    cm.reset()
    cm.add(predictions,y_true)

    score =(2*cm.TP())/(2*cm.TP()+cm.FP()+cm.FN())
    return score


class ROCCurve:
    def __init__(self, y_test, y_pred_score):
        # Init attributes
        self.y_test = y_test
        self.y_pred_score = y_pred_score

    def _get_fpr_tpr(self):
        # thresholds
        thresholds = torch.linspace(0.001, 0.999, 1000).unsqueeze(1)

        # get prediction for all thresholds
        self.y_pred = self.y_pred_score.T > thresholds

        # get TP, FP, TN, and FN for all thresholds
        tp, fp, tn, fn = self._get_tp_fp_tn_fn()

        # calculate true positive rate for all thresholds
        tpr = tp.float() / (tp + fn)

        # calculate false positive rate for all thresholds
        fpr = fp.float() / (fp + tn)

        return fpr.flip((0, )), tpr.flip((0, ))


    def _get_tp_fp_tn_fn(self):

        # change datatype to bool
        self.y_pred = self.y_pred.bool()
        self.y_test = self.y_test.bool()

        # calculate TP
        tp = (self.y_pred & self.y_test).sum(dim=1)

        # calculate FP
        fp = (self.y_pred & ~self.y_test).sum(dim=1)

        # calculate TN
        tn = (~self.y_pred & ~self.y_test).sum(dim=1)

        # calculate FN
        fn = (~self.y_pred & self.y_test).sum(dim=1)

        return tp, fp, tn, fn

    def plot_roc(self):

        # get TPR and FPR and plot TPR-vs-FPR
        plt.plot(*self._get_fpr_tpr(), label='ROC curve', color='g')
        plt.plot([0, 1], [0, 1], label='Random Classifier (AUC = 0.5)', linestyle='--', lw=2, color='r')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.legend(loc="lower right")
        plt.title('ROC Curve')
        plt.show()

    def get_auc_score(self):
        # Get TPR and FPR
        fpr, tpr = self._get_fpr_tpr()

        # get area under the curve of TPR-vs-FPR plot
        return np.trapz(tpr, fpr), fpr, tpr

## <font style="color:green">4. Train and Validation [5 Points]</font>


**Write the methods or classes to be used for training and validation.**

In [ ]:
def train(DEVICE,
          dataset_config: DatasetConfiguration,
          model: nn.Module,
          optimizer: torch.optim.Optimizer,
          train_loader: torch.utils.data.DataLoader,
          epoch_idx: int,
          total_epochs: int,
          weights = None
          ) -> Tuple[float, float]:
    
    # change model in training mode
    model.train()
    acc_metric = MulticlassAccuracy(num_classes=dataset_config.NUM_CLASSES, average="micro")
    mean_metric = MeanMetric()

    status = f"Train:\t{bold}Epoch: {epoch_idx}/{total_epochs}{reset}"

    prog_bar = tqdm(train_loader, bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
    criterion= nn.CrossEntropyLoss(weight=weights,label_smoothing=.1)
    prog_bar.set_description(status)
    for data, target in prog_bar:
        # Send data and target to appropriate device.
        data, target = data.to(DEVICE), target.to(DEVICE)

        lr = optimizer.param_groups[0]['lr']
        # Reset parameters gradient to zero.
        optimizer.zero_grad()
    
        # Forward pass to the model.
        output = model(data)

        # Cross Entropy loss
        loss = criterion(output, target)

        # Find gradients w.r.t training parameters.
        loss.backward()

        # Update parameters using gradients.
        optimizer.step()


        # Batch Loss.
        batch_loss = mean_metric(loss.item(), weight=data.shape[0])

        # Get probability score using softmax.
        prob = F.softmax(output, dim=1)

        # Get the index of the max probability.
        pred_idx = prob.detach().argmax(dim=1)

        # Batch accuracy.
        batch_acc = acc_metric(pred_idx.cpu(), target.cpu())

        # Update progress bar description.
        step_status = status + f"\tLoss: {mean_metric.compute():.4f}, Acc: {acc_metric.compute():.4f}"
        prog_bar.set_description(step_status)

    epoch_loss = mean_metric.compute()
    epoch_acc = acc_metric.compute()

    prog_bar.close()

    return epoch_loss, epoch_acc

In [ ]:
def validate(DEVICE,
    train_config: TrainingConfiguration,
    model: nn.Module,
    test_loader: torch.utils.data.DataLoader,
    epoch_idx: int,
    total_epochs: int
) -> Tuple[float, float]:
    #
    model.eval()

    count_sample = 0
    step_loss = 0
    step_accuracy = 0

    status = f"Valid:\t{bold}Epoch: {epoch_idx}/{total_epochs}{reset}"

    prog_bar = tqdm(test_loader, bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')

    prog_bar.set_description(status)

    for data, target in prog_bar:
        data, target = data.to(DEVICE), target.to(DEVICE)

        # Get the model's predicted logits.
        with torch.no_grad():
            output = model(data)

        # Compute the CE-Loss.
        test_loss = F.cross_entropy(output, target).item()

        # Convert model's logits to probability scores.
        prob = F.softmax(output, dim=1)

        # Get the class id for the maximum score.
        pred_idx = prob.detach().argmax(dim=1)

        # Batch validation loss.
        step_loss+= test_loss * data.shape[0]

        # Batch validation accuracy.
        step_accuracy+= (pred_idx.cpu() == target.cpu()).sum()

        # Count samples.
        count_sample+= data.shape[0]

        # Update progress bar description.
        step_status = status + f"\tLoss: {float(step_loss/count_sample):.4f}, "
        step_status+= f"Acc: {float(step_accuracy/count_sample):.4f}"
        prog_bar.set_description(step_status)


    test_loss = float(step_loss / len(test_loader.dataset))
    test_acc = float(step_accuracy/ len(test_loader.dataset))

    prog_bar.close()

    return test_loss, test_acc

## <font style="color:green">5. Model [5 Points]</font>

**Define your model in this section.**

**You are allowed to use any pre-trained model.**

In [ ]:
def get_resnet_50(num_classes):
    
    model_res_50 = resnet50(weights="DEFAULT")
    
    for params in model_res_50.parameters():
        params.requires_grad = False
    for params in model_res_50.layer3.parameters():
        params.requires_grad = False
    for params in model_res_50.layer4.parameters():
        params.requires_grad = True
     
    model_fc_in_features = model_res_50.fc.in_features

    model_res_50.fc = nn.Sequential(
        nn.Linear(in_features=model_fc_in_features, out_features=128),
        nn.ReLU(),
        nn.Dropout(.4),
        nn.Linear(128,num_classes)
    )
    for params in model_res_50.fc.parameters():
        params.requires_grad = True
    return model_res_50

## <font style="color:green">6. Utils [5 Points]</font>

**Define those methods or classes, which have  not been covered in the above sections.**

In [ ]:
def main(DEVICE: torch.device,
         model: nn.Module,
         optimizer: Union[optim.SGD, optim.Adam],
         ckpt_dir: str,
         summary_writer: torch.utils.tensorboard.writer.SummaryWriter,
         data_config: DatasetConfiguration,
         train_config: TrainingConfiguration,
         weights = None
        ) -> dict:

    # Data loader.
    torch.manual_seed(42) # Set a seed for consistent splits across runs
    # 4. Create DataLoaders for each set
    train_loader,valid_loader = get_data((data_config.IMG_HEIGHT,data_config.IMG_WIDTH),dataset = train_dataset,batch_size =train_config.batch_size )

    best_loss = torch.tensor(np.inf)
    best_acc = torch.tensor(0)
    
    # Epoch train/test loss.
    epoch_train_loss = []
    epoch_valid_loss  = []

    # Epoch train/test accuracy.
    epoch_train_acc = []
    epoch_valid_acc  = []


    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,train_config.EPOCHS)
    # Training time measurement.
    t_begin = time.time()
    for epoch in range(train_config.EPOCHS):
        
        train_loss, train_acc = train(DEVICE,
                                      data_config,
                                      model,
                                      optimizer,
                                      train_loader,
                                      epoch+1,
                                      train_config.EPOCHS,
                                      weights = weights
                                      )
        
        valid_loss, valid_acc = validate(DEVICE,
                                         data_config,
                                         model,
                                         valid_loader,
                                         epoch+1,
                                         train_config.EPOCHS)

        train_loss_stat = f"{bold}Train Loss: {train_loss:.4f}{reset}"
        train_acc_stat  = f"{bold}Train Acc:  {train_acc:.4f}{reset}"

        valid_loss_stat = f"{bold}Valid Loss: {valid_loss:.4f}{reset}"
        valid_acc_stat  = f"{bold}Valid Acc:  {valid_acc:.4f}{reset}"

        print(f"\n{train_loss_stat:<30}{train_acc_stat}")
        print(f"{valid_loss_stat:<30}{valid_acc_stat}")
        
        scheduler.step()
        
        epoch_train_loss.append(train_loss)
        epoch_train_acc.append(train_acc)

        epoch_valid_loss.append(valid_loss)
        epoch_valid_acc.append(valid_acc)
            
        summary_writer.add_scalar('Loss/Train',     train_loss, epoch)
        summary_writer.add_scalar('Accuracy/Train', train_acc, epoch)

        summary_writer.add_scalar('Loss/Validation',     valid_loss, epoch)
        summary_writer.add_scalar('Accuracy/Validation', valid_acc, epoch)

        # Add scalars (loss/accuracy) to tensorboard.
        summary_writer.add_scalars('Loss/train-val',     {'train': train_loss, 'validation': valid_loss}, epoch)
        summary_writer.add_scalars('Accuracy/train-val', {'train': train_acc,  'validation': valid_acc},  epoch)

        if valid_acc > best_acc:
            best_acc = valid_acc
            print(f"\nModel acc Improved... Saving Model ... ", end="")
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), os.path.join(ckpt_dir, "ResNet_TL_Food_acc.pth"))
            print("Done.\n")
        print(f"{'='*72}\n")

    print(f"Total time: {(time.time() - t_begin):.2f}s, Best Loss: {best_loss:.3f}")

    # Load model with best weights.
    model.load_state_dict(best_weights)

    history = dict(model = model,
                   train_loss = epoch_train_loss,
                   train_acc  = epoch_train_acc,
                   valid_loss = epoch_valid_loss,
                   valid_acc  = epoch_valid_acc,
                  )

    return history

In [ ]:
def create_checkpoint_dir(checkpoint_dir):

    # Create a new checkpoint directory every time.
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    print(f"Checkpoint directory: {checkpoint_dir}")
    return checkpoint_dir

## <font style="color:green">7. Experiment [5 Points]</font>

**Choose your optimizer and LR-scheduler and use the above methods and classes to train your model.**

In [ ]:
CKPT_DIR = create_checkpoint_dir(TrainingConfiguration.CHECKPOINT_DIR)

train_config = TrainingConfiguration()
dataset_config = DatasetConfiguration()

In [ ]:
class_counts = train_dataset.label_df["class"].value_counts().sort_index()

print(class_counts)
class_counts = torch.tensor(class_counts)
weights = 1.0 / class_counts
weights = weights / weights.sum()
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

model = get_resnet_50(num_classes=dataset_config.NUM_CLASSES)

print(summary(model,
              input_size=(1, 3, dataset_config.IMG_HEIGHT, dataset_config.IMG_WIDTH),
              row_settings=["var_names"]))


model = model.float().to(DEVICE)


optimizer = optim.AdamW(model.parameters(),
                      lr=train_config.LEARNING_RATE,
                      weight_decay =5e-4
                      )

if DEVICE == 'cuda':
    model = torch.compile(model)

In [ ]:
writer = SummaryWriter(train_config.LOG_DIR)

history = main(DEVICE,
               model = model,
               optimizer = optimizer,
               ckpt_dir = CKPT_DIR,
               summary_writer =  writer,
               data_config = dataset_config,
               train_config = train_config,
               weights = class_weights
              )

writer.close()

## <font style="color:green">8. TensorBoard Log Link [5 Points]</font>

**Share your TensorBoard scalars logs link here You can also share (not mandatory) your GitHub link, if you have pushed this project in GitHub.**


Note: In light of the recent shutdown of tensorboard.dev, we have updated the submission requirements for your project. Instead of sharing a tensorboard.dev link, you are now required to upload your generated TensorBoard event files directly onto the lab. As an alternative, you may also include a screenshot of your TensorBoard output within your Jupyter notebook. This adjustment ensures that your data visualization and model training efforts are thoroughly documented and accessible for evaluation.

You are also welcome (and encouraged) to utilize alternative logging services like wandB or comet. In such instances, you can easily make your project logs publicly accessible and share the link with others.

In [ ]:
def zip_directory(directory_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(directory_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Add file to zip, preserving directory structure within the zip
                zipf.write(file_path, os.path.relpath(file_path, directory_path))

# Example usage:
# Assuming your logs are in a folder named 'logs' within the working directory
log_folder_path = '/kaggle/working/FOOD_DIR'
zip_file_path = '/kaggle/working/FOOD_DIR.zip' # Output zip file will be in the working directory

zip_directory(log_folder_path, zip_file_path)
print(f"Directory '{log_folder_path}' has been zipped to '{zip_file_path}'")

**Share your Kaggle profile link  with us here to score , points in  the competition.**

**For full points, you need a minimum accuracy of `75%` on the test data. If accuracy is less than `70%`, you gain  no points for this section.**


**Submit `submission.csv` (prediction for images in `test.csv`), in the `Submit Predictions` tab in Kaggle, to get evaluated for  this section.**

In [ ]:
import csv
data_config = DatasetConfiguration()
train_confg = TrainingConfiguration()
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "ResNet_TL_Food_acc.pth")))
def predictions_to_csv(test_d,model,output_csv_path):
    model.eval()
    with open(output_csv_path, 'w') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['id','class'])
        for images,image_ids in test_d:
            
            images = images.to(DEVICE)
            
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            
            for img_idx,pred in zip(image_ids,preds.cpu().numpy()):
                writer.writerow([img_idx,idx_to_class[pred]])
    print(f"Predictions saved to {output_csv_path}")
    
test_dataset_loader = get_data((data_config.IMG_HEIGHT,data_config.IMG_WIDTH),dataset=test_dataset, batch_size=train_config.batch_size,test=True)


predictions_to_csv(test_dataset_loader,model,"submission.csv")

Link to Kaggle Competition Results:
https://www.kaggle.com/competitions/open-cv-py-torch-project-2-classification-round-4/leaderboard?